In [1]:
import sqlite3

import pandas as pd
# import pandas as pd

In [9]:
conn = sqlite3.connect('school.db')
cur = conn.cursor()

In [3]:
#create tables

#1. student table
cur.execute("""
CREATE TABLE IF NOT EXISTS students(
    student_id INTEGER PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    email VARCHAR(100) UNIQUE
    )
    """)

#2. COURSE
cur.execute("""
CREATE TABLE IF NOT EXISTS courses(
    course_id INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL,
    lecturer_id INTEGER ,
    FOREIGN KEY(lecturer_id) REFERENCES lecturers(lecturer_id)
    )
    """)
#3. Lecturer
cur.execute("""
CREATE TABLE IF NOT EXISTS lecturers(
    lecturer_id INTEGER PRIMARY KEY,
    lecturer_name VARCHAR(50) NOT NULL,
    email VARCHAR(100) UNIQUE
    )
    """)
#4. Enrolment
cur.execute("""
CREATE TABLE IF NOT EXISTS enrollment(
    enroll_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    course_id INTEGER,
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (course_id) REFERENCES courses(course_id)
    )
    """)
conn.commit()

In [4]:
# Insert students
cur.executemany("""
INSERT INTO students VALUES (?, ?, ?)
""", [
    (1, "Mary Wanjiku", "mary@example.com"),
    (2, "John Kamau", "john@example.com"),
    (3, "Grace Achieng", "grace@example.com"),
    (4, "Peter Otieno", "peter@example.com"),
    (5, "Anne Njeri", "anne@example.com")
])
conn.commit()

In [5]:
# Insert lecturers
cur.executemany("""
INSERT INTO lecturers VALUES (?, ?, ?)
""", [
    (1, "Dr. Mwangi", "mwangi@gmail.com"),
    (2, "Ms. Amina", "amina12@gmail.com"),
    (3, "Mr. Otieno", "oti754eno@gmail.com")
])
conn.commit()

In [6]:
# Insert courses
cur.executemany("""
INSERT INTO courses VALUES (?, ?, ?)
""", [
    (101, "Python Programming", 1),
    (102, "SQL and Databases", 2),
    (103, "Machine Learning", 1),
    (104, "Data Visualization", 3)
])
conn.commit()

In [7]:
# Insert enrollments
cur.executemany("""
INSERT INTO enrollment VALUES (?, ?, ?)
""", [
    (1, 1, 101),
    (2, 1, 102),
    (3, 2, 101),
    (4, 2, 103),
    (5, 3, 102),
    (6, 4, 104)
])

conn.commit()

In [10]:
# closing connection to use pd.read_sql for easier quering
conn.close()

## 1.Inner join-
select students who have courses enrolled

In [11]:
conn = sqlite3.connect("school.db")

## 2.output all our students Plus their courses
(either enrolled or not)

In [13]:
q = """
SELECT s.name, c.course_name
FROM students AS s
INNER JOIN enrollment AS e
    ON s.student_id = e.student_id
INNER JOIN courses AS c
    ON e.course_id = c.course_id
    """

cur.execute(q)
result = cur.fetchall()

for row in result:
    print(row)

('Mary Wanjiku', 'Python Programming')
('Mary Wanjiku', 'SQL and Databases')
('John Kamau', 'Python Programming')
('John Kamau', 'Machine Learning')
('Grace Achieng', 'SQL and Databases')
('Peter Otieno', 'Data Visualization')


## 3.Find :
student, course, lecturer(& grade) in one query

In [15]:
q= """
SELECT s.name, c.course_name, l.lecturer_name
FROM students AS s
INNER JOIN enrollment AS e
    ON s.student_id = e.student_id
INNER JOIN courses AS c
    ON c.course_id = e.course_id
INNER JOIN lecturers AS l
    ON c.lecturer_id = l.lecturer_id
"""

df= pd.read_sql_query(q, conn)
print(df)

            name         course_name lecturer_name
0   Mary Wanjiku  Python Programming    Dr. Mwangi
1   Mary Wanjiku   SQL and Databases     Ms. Amina
2     John Kamau  Python Programming    Dr. Mwangi
3     John Kamau    Machine Learning    Dr. Mwangi
4  Grace Achieng   SQL and Databases     Ms. Amina
5   Peter Otieno  Data Visualization    Mr. Otieno


In [12]:
pd.read_sql("""
SELECT
   l.lecturer_name, c.course_name
FROM lecturers AS l
INNER JOIN courses AS c
   ON l.lecturer_id = c.lecturer_id;
""", conn)


,lecturer_name,course_name
0,Dr. Mwangi,Python Programming
1,Ms. Amina,SQL and Databases
2,Dr. Mwangi,Machine Learning
3,Mr. Otieno,Data Visualization


In [13]:
pd.read_sql("""
SELECT l.lecturer_name, c.course_name
FROM lecturers AS l
LEFT JOIN courses AS c ON l.lecturer_id = c.lecturer_id;
""", conn)

,lecturer_name,course_name
0,Dr. Mwangi,Machine Learning
1,Dr. Mwangi,Python Programming
2,Ms. Amina,SQL and Databases
3,Mr. Otieno,Data Visualization


In [21]:
pd.read_sql("""
SELECT *
FROM students AS s
LEFT JOIN enrollment AS e
   ON s.student_id = e.student_id
WHERE e.course_id IS NULL;
""", conn)

,student_id,name,email,enroll_id,student_id,course_id
0,5,Anne Njeri,anne@example.com,None,None,None


In [28]:
pd.read_sql("""
SELECT 
    c.course_name, 
    COUNT(e.student_id) AS student_count
FROM courses AS c
LEFT JOIN enrollment AS e 
    ON c.course_id = e.course_id
    GROUP BY c.course_name;
""", conn)

,course_name,student_count
0,Data Visualization,1
1,Machine Learning,1
2,Python Programming,2
3,SQL and Databases,2


In [30]:
# Block of code to add grades column in ENROLLMENT table
conn.close()

conn = sqlite3.connect('school.db')
cur = conn.cursor()

# Add the new column (e.g., 'grade' with TEXT type)
cur.execute("ALTER TABLE enrollment ADD COLUMN grade TEXT")
conn.commit()

# List of tuples: (new_value, identifier_value)
grades_data = [
    ("A", 1),
    ("B", 2),
    ("C", 3),
    ("B", 4),
    ("A", 5),
]

# Update the new column where the row matches the primary key/ID
cur.executemany(
    """
    UPDATE enrollment
    SET grade = ?
    WHERE enroll_id = ?
""",
    grades_data,
)

conn.commit()
# Inspect table structure
for col in cur.execute("PRAGMA table_info(enrollment)"):
    print(col)

# View the updated data
for row in cur.execute("SELECT * FROM enrollment LIMIT 5"):
    print(row)

conn.close()

(0, 'enroll_id', 'INTEGER', 0, None, 1)
(1, 'student_id', 'INTEGER', 0, None, 0)
(2, 'course_id', 'INTEGER', 0, None, 0)
(3, 'grade', 'TEXT', 0, None, 0)
(1, 1, 101, 'A')
(2, 1, 102, 'B')
(3, 2, 101, 'C')
(4, 2, 103, 'B')
(5, 3, 102, 'A')


In [ ]:
conn = sqlite3.connect("school.db")

pd.read_sql("""
SELECT s.name, c.course_name, l.lecturer_name, e.grade
FROM students AS s
INNER JOIN enrollment As e
   ON s.student_id = e.student_id
INNER JOIN courses AS c
   ON c.course_id = e.course_id
INNER JOIN lecturers AS l
   ON c.lecturer_id = l.lecturer_id
   ORDER BY s.name;
""", conn)

,name,course_name,lecturer_name,grade
0,Grace Achieng,SQL and Databases,Ms. Amina,A
1,John Kamau,Python Programming,Dr. Mwangi,C
2,John Kamau,Machine Learning,Dr. Mwangi,B
3,Mary Wanjiku,Python Programming,Dr. Mwangi,A
4,Mary Wanjiku,SQL and Databases,Ms. Amina,B
5,Peter Otieno,Data Visualization,Mr. Otieno,NaN


In [36]:
pd.read_sql("""
SELECT DISTINCT s.name, c.course_name
FROM students AS s
INNER JOIN enrollment AS e
   ON s.student_id = e.student_id
INNER JOIN courses AS c
   ON c.course_id = e.course_id
INNER JOIN lecturers AS l
   ON c.lecturer_id = l.lecturer_id
WHERE l.lecturer_name = 'Dr. Mwangi';
""", conn)

,name,course_name
0,Mary Wanjiku,Python Programming
1,John Kamau,Python Programming
2,John Kamau,Machine Learning


In [42]:
pd.read_sql("""
SELECT 
    s.name, 
    COUNT(e.course_id) AS course_count
FROM students AS s
LEFT JOIN enrollment AS e 
   ON s.student_id = e.student_id
GROUP BY c.course_name;
HAVING COUNT(e.course_id) > 1;
""", conn)

DatabaseError: Execution failed on sql '
SELECT 
    s.name, 
    COUNT(e.course_id) AS course_count
FROM students AS s
LEFT JOIN enrollment AS e 
   ON s.student_id = e.student_id
GROUP BY c.course_name;
HAVING COUNT(e.course_id) > 1;
': no such column: c.course_name